# Resistive decay and convergence

Vary toroidal diffusivity, spatial resolution and timestep in the coupled shell model.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from nsinduct.shell import (
    make_grid,
    run_case,
    volume_rms,
)

base = dict(
    ntheta=16,
    nr=24,
    r_inner=0.3,
    r_outer=1.0,
    field_strength=0.07,
    eta=0.0,
    nu=0.0,
    timestep=0.01,
    output_dt=0.1,
    snapshot_count=3,
    stop_time=12.0,
    lorentz_feedback=True,
)

In [ ]:
etas = [0.0, 2e-5, 1e-4, 5e-4]
runs = [run_case(base | dict(eta=e, stop_time=16.0, name=f"eta{e}")) for e in etas]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
for e, run in zip(etas, runs):
    rr = run["rows"]
    tt = [r["time"] for r in rr]
    ax[0].plot(tt, [r["E_tor"] for r in rr], label=str(e))
    ax[1].plot(tt, [r["cum_ohmic"] for r in rr], label=str(e))
    print(
        "eta, peak energy, sampled peak time, final normalized budget:",
        e,
        run["summary"]["peak_E_tor"],
        run["summary"]["time_peak_E_tor"],
        rr[-1]["cumulative_energy_residual"],
    )
ax[0].set(xlabel=r"$t$", ylabel="toroidal magnetic energy")
ax[0].legend(title="eta")
ax[1].set(xlabel=r"$t$", ylabel="integrated Ohmic loss")
plt.show()

In [ ]:
spatial = []
for n in [16, 24, 32]:
    cfg = base | dict(
        ntheta=n,
        nr=3 * n // 2,
        stop_time=2.0,
        timestep=0.005,
        snapshot_count=2,
        name=f"grid{n}",
    )
    run = run_case(cfg)
    g = make_grid(cfg)
    snap = run["snapshots"][-1]["fields"]
    spatial.append(
        (
            n,
            run["rows"][-1]["E_tor"],
            volume_rms(g, g.s * snap["q"]),
            np.max(np.abs(g.s * snap["q"])),
        )
    )
print("ntheta, E_tor(t=2), RMS Bphi, max |Bphi|")
for row in spatial:
    print(row)
temporal = []
for dt in [0.08, 0.04, 0.02, 0.01]:
    run = run_case(
        base
        | dict(
            timestep=dt, stop_time=2.0, output_dt=2.0, snapshot_count=2, name=f"dt{dt}"
        )
    )
    temporal.append((dt, run["rows"][-1]["E_tor"], run["snapshots"][-1]["fields"]["q"]))
g = make_grid(base)
print("dt, energy error against dt=.01, relative weighted q difference")
for dt, energy, q in temporal[:-1]:
    print(
        dt,
        abs(energy - temporal[-1][1]),
        volume_rms(g, g.s * (q - temporal[-1][2]))
        / volume_rms(g, g.s * temporal[-1][2]),
    )

Stronger diffusion reduces and advances the toroidal-energy peak. Local extrema are more resolution-sensitive than integral energies. Timestep errors are measured against the finest numerical run.

## Spectral resistive eigenmode

For zero flow and poloidal field, $q=[j_1(kr)+a y_1(kr)]/r$ vanishes at both radial walls and decays as $\exp(-\eta k^2t)$.

In [ ]:
import logging

logging.getLogger("dedalus").setLevel(logging.WARNING)

from scipy.special import spherical_jn, spherical_yn
from scipy.optimize import brentq
from nsinduct.dedalus_backend import run_case as run_spectral

def determinant(k):
    return spherical_jn(1, 0.3*k)*spherical_yn(1, k) - spherical_yn(1, 0.3*k)*spherical_jn(1, k)

k_mode = brentq(determinant, 4.0, 6.0)
def q_mode(theta, r):
    radial = (spherical_jn(1, k_mode*r) - spherical_jn(1, 0.3*k_mode)/spherical_yn(1, 0.3*k_mode)*spherical_yn(1, k_mode*r))/r
    return np.broadcast_to(radial, np.broadcast_shapes(np.shape(theta), np.shape(r))).copy()

errors = []
for nt, nr in [(12, 18), (16, 24)]:
    cfg = dict(ntheta=nt, nr=nr, field_strength=0.0, eta=0.01, nu=0.0,
               lorentz_feedback=False, stop_time=0.2, timestep=0.002,
               output_dt=0.1, snapshot_count=2,
               initial_omega=lambda theta, r: np.zeros(np.broadcast_shapes(np.shape(theta), np.shape(r))),
               initial_q=q_mode)
    decay = run_spectral(cfg)
    snap = decay["snapshots"][-1]
    r, theta = np.meshgrid(snap["fields"]["radius"], snap["fields"]["theta"])
    exact = q_mode(theta, r)*np.exp(-0.01*k_mode**2*snap["time"])
    error = np.max(abs(snap["fields"]["q"] - exact))/np.max(abs(exact))
    errors.append(error)
    print(f"{nt}×{nr}: relative maximum q error = {error:.3e}; radial wall residual = {decay['summary']['max_toroidal_wall_residual']:.3e}")
assert max(errors) < 1e-5
plt.semilogy([18, 24], errors, "o-")
plt.xlabel(r"$N_r$")
plt.ylabel("Relative maximum decay error")
plt.show()
